# Défi quotidien : Prétraitement et optimisation des modèles basés sur les transformateurs

Jeu de données : **Contradictory, My Dear Watson** — des paires *(premise, hypothesis)* multilingues à classer en 3 catégories :
- `0` = entailment (implication)
- `1` = neutral (neutre)
- `2` = contradiction (contradiction)

Ce notebook couvre : tokenisation BERT / XLM-RoBERTa, préparation des entrées, exploration du dataset, et validation croisée k-fold.

In [ ]:
# Installation des dépendances (décommentez si nécessaire)
# %pip install -q transformers torch scikit-learn pandas


## 1. Comprendre BERT et XLM-RoBERTa

**BERT** (*Bidirectional Encoder Representations from Transformers*) est un modèle encodeur entraîné sur de l'anglais (principalement) avec deux objectifs de pré-entraînement : *Masked Language Modeling* (prédire des tokens masqués) et *Next Sentence Prediction*. Il utilise un tokenizer **WordPiece** et des tokens spéciaux `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`.

**XLM-RoBERTa** est une variante multilingue de RoBERTa (lui-même une version optimisée de BERT, sans NSP, entraînée plus longtemps avec plus de données). XLM-RoBERTa est pré-entraîné sur 100 langues simultanément avec un tokenizer **SentencePiece**, ce qui le rend particulièrement adapté à notre jeu de données multilingue. Ses tokens spéciaux sont `<s>` (début de séquence), `</s>` (fin de séquence), `<pad>`, `<mask>`.

Versions pré-entraînées courantes :
- `bert-base-uncased`, `bert-base-multilingual-cased`
- `xlm-roberta-base`, `xlm-roberta-large`

Étant donné que notre dataset contient 15 langues différentes (anglais, arabe, chinois, français, hindi, etc.), **XLM-RoBERTa** est le choix le plus pertinent ici, mais nous explorerons les deux tokenizers.

In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

print("BERT vocab size:", bert_tokenizer.vocab_size)
print("XLM-R vocab size:", xlmr_tokenizer.vocab_size)
print("BERT special tokens:", bert_tokenizer.special_tokens_map)
print("XLM-R special tokens:", xlmr_tokenizer.special_tokens_map)


## 2. Tokenisation du texte

Nous utilisons `encode_plus()` pour transformer une (ou deux) phrase(s) en `input_ids`, `attention_mask` et, si on le souhaite, des `token_type_ids`. Nous testons d'abord avec une phrase unique, puis avec une paire *(premise, hypothesis)* comme dans notre tâche de NLI.

In [ ]:
# Tokenisation d'une phrase unique
single_sentence = "BERT and XLM-RoBERTa are powerful transformer models."

encoded_single = bert_tokenizer.encode_plus(
    single_sentence,
    add_special_tokens=True,
    return_attention_mask=True,
    return_tensors="pt"
)

print("input_ids:", encoded_single["input_ids"])
print("attention_mask:", encoded_single["attention_mask"])
print("Tokens:", bert_tokenizer.convert_ids_to_tokens(encoded_single["input_ids"][0]))
print("Decoded:", bert_tokenizer.decode(encoded_single["input_ids"][0]))


In [ ]:
# Tokenisation d'une paire de phrases (premise, hypothesis) -> tâche de NLI
premise = "A man is playing a guitar on stage."
hypothesis = "A man is performing music."

encoded_pair = bert_tokenizer.encode_plus(
    premise,
    hypothesis,
    add_special_tokens=True,
    return_attention_mask=True,
    return_token_type_ids=True,
    return_tensors="pt"
)

print("input_ids:", encoded_pair["input_ids"])
print("token_type_ids:", encoded_pair["token_type_ids"])
print("attention_mask:", encoded_pair["attention_mask"])
print("Tokens:", bert_tokenizer.convert_ids_to_tokens(encoded_pair["input_ids"][0]))
print("Decoded:", bert_tokenizer.decode(encoded_pair["input_ids"][0]))

# Pour la classification, le label associé (entailment/neutral/contradiction) sera fourni séparément
# par le dataset (colonne 'label'), il ne fait pas partie de l'encodage du tokenizer.
example_label = 0  # 0 = entailment, 1 = neutral, 2 = contradiction
print("\nExemple de label associé:", example_label)


In [ ]:
# Même exemple avec XLM-RoBERTa, pour comparer les tokens spéciaux et le découpage
encoded_pair_xlmr = xlmr_tokenizer.encode_plus(
    premise,
    hypothesis,
    add_special_tokens=True,
    return_attention_mask=True,
    return_tensors="pt"
)

print("Tokens XLM-R:", xlmr_tokenizer.convert_ids_to_tokens(encoded_pair_xlmr["input_ids"][0]))
print("Decoded XLM-R:", xlmr_tokenizer.decode(encoded_pair_xlmr["input_ids"][0]))


### Réflexion
- `input_ids` : les identifiants numériques des tokens dans le vocabulaire du modèle.
- `attention_mask` : indique au modèle quels tokens sont réels (1) et lesquels sont du remplissage (0), afin que le self-attention les ignore.
- `token_type_ids` (BERT uniquement) : distingue la première phrase (0) de la seconde (1) dans une paire — XLM-RoBERTa n'utilise pas cette information de la même façon, il s'appuie plutôt sur les tokens `</s></s>` pour séparer les phrases.
- BERT utilise `[CLS]` / `[SEP]`, alors que XLM-RoBERTa utilise `<s>` / `</s>`.

## 3. Préparation des données d'entrée pour le modèle

Pour entraîner un modèle, toutes les séquences d'un batch doivent avoir la **même longueur**. On utilise donc le *padding* (ajout de tokens `[PAD]`/`<pad>`) et la *truncation* (coupe des séquences trop longues) avec une `max_length` fixe.

In [ ]:
max_length = 64

def encode_example(tokenizer, premise, hypothesis, max_length=max_length):
    return tokenizer.encode_plus(
        premise,
        hypothesis,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_attention_mask=True,
        return_tensors="pt"
    )

encoded_padded = encode_example(bert_tokenizer, premise, hypothesis)
print("Shape input_ids:", encoded_padded["input_ids"].shape)
print("input_ids:", encoded_padded["input_ids"][0])
print("attention_mask:", encoded_padded["attention_mask"][0])

print("\nSpecial tokens map (BERT):", bert_tokenizer.special_tokens_map)
print("Vocab size (BERT):", bert_tokenizer.vocab_size)
print("\nSpecial tokens map (XLM-R):", xlmr_tokenizer.special_tokens_map)
print("Vocab size (XLM-R):", xlmr_tokenizer.vocab_size)


**Pourquoi l'`attention_mask` est-il important ?** Sans lui, le modèle traiterait les tokens `[PAD]` comme du contenu réel et leur attribuerait de l'attention, ce qui fausserait les représentations. L'`attention_mask` met une valeur de `0` sur ces positions, ce qui force le mécanisme de self-attention à leur attribuer un poids quasi nul (en pratique, un grand nombre négatif est ajouté aux scores d'attention avant le softmax).

## 4. Chargement et exploration de l'ensemble de données

In [ ]:
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


In [ ]:
test_df.head()


In [ ]:
print("Colonnes train:", list(train_df.columns))
print("Colonnes test:", list(test_df.columns))

print("\nValeurs manquantes (train):\n", train_df.isna().sum())

print("\nRépartition des labels (0=entailment, 1=neutral, 2=contradiction):")
print(train_df["label"].value_counts())

print("\nRépartition des langues:")
print(train_df["language"].value_counts())


Les colonnes nécessaires à l'entraînement sont : `premise`, `hypothesis` (les textes d'entrée) et `label` (la cible à prédire). Les colonnes `lang_abv` et `language` peuvent servir à analyser les performances par langue.

## 5. Création de plis de validation croisée (Stratified K-Fold)

On utilise `StratifiedKFold` pour garantir que chaque pli (*fold*) conserve la même proportion de chaque classe de `label`, ce qui est important car nos 3 classes ne sont pas parfaitement équilibrées.

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

N_SPLITS = 5
RANDOM_STATE = 42

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

X = train_df["premise"] + " [SEP] " + train_df["hypothesis"]  # texte combiné, utilisé seulement pour le split
y = train_df["label"]

train_folds = []
val_folds = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_df.iloc[val_idx].reset_index(drop=True)

    train_folds.append(fold_train_df)
    val_folds.append(fold_val_df)

    print(f"Fold {fold_idx}: train={len(fold_train_df)}, val={len(fold_val_df)}")
    print("  Répartition labels (train):", fold_train_df['label'].value_counts(normalize=True).round(3).to_dict())
    print("  Répartition labels (val):  ", fold_val_df['label'].value_counts(normalize=True).round(3).to_dict())


In [ ]:
# Vérification : la répartition globale est bien préservée dans chaque fold
print("Répartition globale des labels:")
print(train_df["label"].value_counts(normalize=True).round(3))


## Prochaines étapes (hors périmètre de ce notebook)

- Construire un `Dataset`/`DataLoader` PyTorch encodant `premise` + `hypothesis` avec `encode_plus()` et `padding="max_length"`.
- Charger `AutoModelForSequenceClassification` (`xlm-roberta-base`, `num_labels=3`).
- Entraîner et valider sur chacun des 5 folds, puis moyenner les métriques (accuracy / F1) pour obtenir une estimation robuste de la performance.
- Optionnel : moyenne (*ensembling*) des prédictions des 5 modèles entraînés sur chaque fold pour la soumission finale.